# gotooltrain — 4B tam fine-tune (Colab)

`Qwen/Qwen3.5-4B` → Go + tool-use modeli, tam fine-tune, **periyodik Hugging Face Hub
yüklemesiyle**.

Bu defter şunu yapar, başka hiçbir şeyi:

1. Depoyu remote'dan çeker (`alexzakarov/model-trainer`).
2. `pip install -e ".[dev,train]"` ile kurar.
3. **Go ve rtk toolchain'lerini** sabitlenmiş digest'lerle kurar (Colab'da ikisi de yok).
4. Go-UT-Bench korpusunu indirip ölçer (lisans süzgeci dahil).
5. Token formatını **gerçek Qwen3.5-4B tokenizer'ıyla** doğrular — maske yanlışsa
   burada durur, GPU saatleri harcanmaz.
6. 4B tam fine-tune'u başlatır ve **her N adımta** checkpoint'ı Hub'a gönderir.

---

## ⚠️ `REPO_REF`'i sabitle

`REPO_REF` şu an `main`. Bu **bir kez çalışmak** içindir: yarın aynı defteri
çalıştırmak farklı kod demektir. `95b1aa3` gibi bir commit sha'sına sabitle,
böylece aynı komut aynı kodu çeker.

## ⚠️ Token'ın yazma yetkisi

Hedef depo 1. hücredeki `HF_REPO_ID`. `HF_TOKEN`'ın o depoya **yazma** yetkisi
olmalı. Depo şu an boş; ilk push onu doldurur. `upload_folder` model card yazmaz
— kart elle eklenir.

## ⚠️ GPU seçimi

4B **tam** fine-tune AdamW ile ~18 GB (ağırlık + gradyan) ister. Colab'da:

| Runtime | VRAM | Bu defter |
|---|---|---|
| T4 (ücretsiz) | 15 GB | ❌ çalışmaz |
| L4 | 24 GB | ⚠️ Adafactor + kısa bağlamla sığar |
| **A100 40 GB** | 40 GB | ✅ önerilen |
| A100 80 GB | 80 GB | ✅ |

Aşağıdaki hücre yetersiz VRAM'i sessizce geçmez — adını ve nedenini söyler.

In [ ]:
# @title 1 — Parameters
#
# Buradaki değerler mock olarak girdi. Gerçek koşuda HF_TOKEN'ı Colab Secrets'a
# (🔑 soldaki panel) koy; buraya yazmak token'ı notebook çıktısına ve git geçmişine
# sızdırır.

REPO_URL = 'https://github.com/alexzakarov/model-trainer.git'
REPO_REF = "main"                      # commit sha da olabilir; sha daha tekrarlanabilir

MODEL_ID = "Qwen/Qwen3.5-4B"          # eğitilecek taban model
HF_REPO_ID = 'alexzakkarov/qwen3.5-golang'    # hedef Hub deposu
HF_TOKEN = "hf_mock_replace_me"        # mock — Colab Secrets'tan okunacak
DRY_RUN = False                        # True: takvimi prova et, hiçbir şey yükleme

# Eğitim
#
# Bağlam ve bellek bütçesi birlikte seçilir; ikisi de ölçülmüş değerlerdir, devir
# değil. Qwen3.5-4B'de 32 katmanın 24'ü Gated DeltaNet (lineer dikkat) ve durum
# [32 v_heads, 128, 128] yani **token başına 1 MB**. Bu yüzden aktivasyon belleği
# bağlamla doğrusal büyür ve 4B'nin kalıcı tabanı (ağırlık + gradyan + adafactor)
# **17,34 GB**'dır — ne yaparsan yap geri kalanı daraltamazsın.
#
# Gönderilen tahmin (gerçek config, ölçülmüş %32 denetimli oran):
#
#     bağlam   resident  transient   loss    toplam   40 GB'de
#      2048      17,34      2,08    0,61     20,34     evet
#      4096      17,34      4,16    1,21     23,33     evet
#      8192      17,34      8,31    2,43     29,33     evet
#     16384      17,34     16,62    4,85     41,32     HAYIR
#
# 8192 seçildi: 40 GB bütçesinin içinde ~10 GB pay bırakıyor, korpusun %72'sini
# kullanıyor. (Aktivasyon terimi *tahmindir*; kalıcı olanlar kesin. 10. hücre
# bütçeyi basar, hangi terimin belirsiz olduğunu orada görürsün.)
#
# 10. hücre `--memory-budget-gb` ile bu tabloyu zorlar: sığmıyorsa **başlamadan**
# durur. 16384 isteyen 40 GB'lık bir kartta reddedilecek, 80 GB'lıkta geçecek.
CONTEXT_LENGTH = 8192
MEMORY_BUDGET_GB = 40.0

# Kaldığın yerden devam?
#
# Boş bırakırsan **taban modelden** başlarsın: Qwen/Qwen3.5-4B, sıfırdan.
# Bu, ilk koşu için doğru olan.
#
# Hugging Face'te bu projeden çıkmış bir checkpoint varsa buraya o repo id'sini yaz
# (yukarıdaki HF_REPO_ID ile aynısı), ağırlıklar oradan yüklenir. Ama neyi geri
# getirdiğini bil: yayınlanan klasör yalnızca `save_pretrained` çıktısıdır —
# ağırlık ve tokenizer. **Optimizer durumu, scheduler, adım sayacı ve push
# geçmişi saklanmaz.** Yani:
#
#   * Adafactor'ın momentleri sıfırdan başlar → ilk birkaç adımın gradyanı
#     daha büyük, bir an için geri gidebilirsin.
#   * Adım sayacı 1'den başlar → daha önce gördüğü örnekleri **ikinci kez** görürsün.
#   * Yayın geçmişi boş sayılır → ilk push yeni sayılır.
#
# Bu bir "kaldığı yerden devam" değil, "o ağırlıklarla yeniden eğitim".
# Gerçek devam için optimizer durumu da saklanmalı; şu an saklanmıyor.
#
# DİKKAT: boş bırakırsan ve repoda zaten bir checkpoint varsa, ilk push'ta (25.
# adım) o checkpoint'in üzerine **yazılır** — 25 adım ilerlemiş taban model
# göndereceksin. Geri dönüşü olmayan bir hata. Varsayılan "auto" bunu kendiliğinden
# çözer: 5. hücre repoyu sorar, ağırlık varsa devam eder, yoksa sıfırdan başlar.
# Yine de yazılıyor, çünkü bu hücrenin ne yapacağını okumadan çalıştırmak da
# bir tercih, ve 12. hücre ne gönderileceğini yine de söyler.

RESUME_FROM = "auto"   # "auto" = repoda checkpoint varsa devam et | "" = bilerek
                       # sıfırdan | bir repo id ya da revizyon = elle seç

# Tercih optimizasyonu (12-13. hücreler) için her görevin kaç kez deneneceği.
# 1 OLABİLMEZ: tercih, birinin geçip birinin kaldığı iki denemeden doğar. Tek
# denemede "başarılı/başarısız" çifti yoktur, `preferences` boş döner.
N_SAMPLES = 4

# --- boru hattı ayarları ---------------------------------------------------
#
# Bunlar 2. hücredeki tek komuta gider; sıra ve koşullar `pipeline.py` içinde.

OPTIMIZER = "adafactor"
SFT_OUTPUT = "runs/colab-sft"

# Tercih aşaması, görev dosyası (`data/eval/tasks.jsonl`) ve ornekleme yapacak
# çalışan bir model gerektirir. O katman kurulana kadar False: atlanan aşama
# *atlandığını* söyler, sahte bir başarı üretmez.
RUN_PREFERENCE_STAGE = False

# Kalite kapısı her koşuda yeniden kurulmaz; dakikalar verir ve sonucu
# değiştirmez. Kapatmak bir tercih olduğu için burada adı var.
RUN_QUALITY_GATE = True

EXPANDABLE_SEGMENTS = True               # parçalanmayı azaltır (bkz. 4. hücre)
EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 1e-5
MAX_RECORDS = 400                      # korpus 4491 kayıt; süreyi sınırlamak için kes
MAX_TOKENS_PER_RECORD = CONTEXT_LENGTH  # üstünü atılır, *sayısı raporlanır*

# Yayınlama
PUSH_EVERY = 25                        # optimizer adımı

print(f"repo   : {REPO_URL}@{REPO_REF}")
print(f"model  : {MODEL_ID}")
print(f"push to: {HF_REPO_ID} every {PUSH_EVERY} steps (dry_run={DRY_RUN})")
print(f"resume : {RESUME_FROM or 'yok - taban modelden sıfırdan'}")

In [ ]:
# @title 2 — Depoyu çek ve kur
#
# REF bir commit sha ise tam tekrarlanabilir olur. 'main' daha basit ama yarın
# aynı defteri çalıştırmak farklı kod demektir.

import importlib
import os
import subprocess
import sys

os.environ["GIT_TERMINAL_PROMPT"] = "0"  # şifre soran bir clone sessizce beklemesin

def run(*args: str) -> str:
    """Bir komutu çalıştır; stdout döndür, hata olursa mesajla dur."""
    result = subprocess.run(args, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        raise SystemExit(f"{' '.join(args)} failed:\n{result.stdout}\n{result.stderr}")
    return result.stdout.strip()

target = "/content/model-trainer"
if os.path.isdir(target):
    run("git", "-C", target, "fetch", "--all")
    run("git", "-C", target, "checkout", REPO_REF)
    run("git", "-C", target, "pull", "--ff-only", "origin", REPO_REF)
else:
    run("git", "clone", REPO_URL, target)
    run("git", "-C", target, "checkout", REPO_REF)

print(run("git", "-C", target, "log", "-1", "--format=%H %s"))

# transformers 5.x şart: Qwen3.5 config'i model_type qwen3_5 bildiriyor ve 4.x onu
# reddediyor (tokenizer yüklenir, config yüklenmez). Sürüm burada da sabitleniyor,
# çünkü Colab'ın imajı zamanla değişir.
run(sys.executable, "-m", "pip", "install", "-q", "-U", "pip")
run(sys.executable, "-m", "pip", "install", "-q", "transformers>=5.17,<6")
run(sys.executable, "-m", "pip", "install", "-q", "-e", f"{target}[dev,train]")

os.chdir(target)
print("cwd:", os.getcwd())

# ---------------------------------------------------------------- kernel path
#
# `pip install -e` yalnızca *yeni* süreçlerde görünür. .pth ve editable-finder
# çengelleri yorumlayıcı **başlangıcında** kurulur; Colab çekirdeği ise bu
# hücreden çok önce başlamıştır. Yani alt süreçler (kalite kapısı, eğitim koşusu)
# paketi görür, çekirdek görmez — ve hata beş hücre sonra, başka bir hücrenin
# satırında belirir. Burada ölçtüm: pip'i çalıştıran yorumlayıcı kendi kurduğu
# paketi import edemiyor.
#
# Bu yüzden kaynak dizini çekirdeğin sys.path'ine de ekleniyor. Gizli bir
# düşüş değil: kurulum zaten yapıldı, bu yalnızca aynı kurulumu *bu* sürece
# görünür kılıyor. Konsol betikleri (gotooltrain-train vb.) alt süreçlerde çalışır
# ve zaten yolunu bulur.
sys.path.insert(0, os.path.join(target, "src"))

# Ama bir tuzak daha var ve bu seviyesiz: `import`, `sys.modules`'tan önbelleklenir.
# Bu hücreyi **ikinci kez** çalıştırırsan kod yeni (pull edildi) ama çekirdek
# eski modülü kullanmaya devam ediyor. Ölçtüm: pull sonrası `import` hâlâ bir
# önceki sürümü döndürüyor. Sonucu görmek kolay: 8192 bağlamı yine OOM eder —
# ama bu kez "40 GB'a sığdırdık" diye düşünürsün, çünkü bellek bütçesi hücrede
# güzel görünüyor. O yüzden önbellek düşürülüyor; `invalidate_caches()` olmadan
# silmek yetmiyor, çünkü bayt-kod önbelleği ayrı tutuluyor.
for _name in [n for n in list(sys.modules) if n == "gotooltrain" or n.startswith("gotooltrain.")]:
    del sys.modules[_name]
importlib.invalidate_caches()

import gotooltrain  # noqa: E402

print("gotooltrain:", gotooltrain.__version__, "->", gotooltrain.__file__)

# Klonun içinden geldiğini doğrula. Path'te başka bir gotooltrain varsa (sık
# gelen bir tuzak: daha önce kurulmuş bir sürüm) sessizce onu eğitirdik.
if not gotooltrain.__file__.startswith(target):
    raise SystemExit(
        f"gotooltrain {gotooltrain.__file__} konumundan geliyor, klon {target} değil. "
        "Sistemde başka bir kurulum var ve eğitimi yanlış kodla yapacağız."
    )

# Koşunun ihtiyaç duyduğu yetenekler gerçekten var mı? Kontrolün kendisi
# import: burada hata veren bir koşu zaten yapamayacağı bir şeyi denemeye
# çalışıyordu, ve iki saat sonra OOM olarak dönecekti. `REPO_REF` eski bir
# revizyona sabitlenmişse koşu burada, açık bir mesajla durur.
#
# `from ... import` bilinçli seçildi: `gotooltrain.train` bir *fonksiyon* adıyla
# da var (paket `train` işlevini dışa aktarıyor) ve modülü gölgeliyor. Ölçtüm:
# `hasattr(gotooltrain.train, ...)` üç yeteneğin de False'unu veriyor, `import
# ... as` da fonksiyonu bağlıyor. Yani ilk yazdığım kontrol, elindeki kod
# doğru olsa bile "eksik" diyordu — doğru bir koşuyu durdurup hatayı koda
# yazıyordu. from-import modülü sys.modules üzerinden çözer, gölgelenmez.
try:
    # Kullanılmıyorlar; buradaki import'un kendisi sözleşme. noqa: F401.
    from gotooltrain.train import (  # noqa: E402, F401
        DEFAULT_MEMORY_BUDGET_GB,
        completion_only_loss,
        enter_training_mode,
    )
except ImportError as _exc:
    raise SystemExit(
        f"{REPO_REF} revizyonu bu koşunun ihtiyaç duyduğu yeteneklerden yoksun "
        f"({_exc.name}). Bellek bütçesinin işe yaraması bunlar olmadan mümkün "
        "değil; yeni bir revizyona sabitle ya da Runtime'ı yeniden başlat."
    ) from _exc
print(f"40 GB koşusunun yetenekleri: tamam (bütçe {DEFAULT_MEMORY_BUDGET_GB:.0f} GB)")

In [ ]:
# @title 3 — Go ve rtk toolchain'leri
#
# Colab'da ikisi de yok. Bu bir eksik değil, **kapının kırılma biçimi**: katalogdaki
# komutların çoğu rtk üzerinden çalışır (`rtk go test`, `rtk read`, `rtk grep`), ve
# paketin kendi testleri gerçek komutları koşar. rtk yokken `go_test` "rtk not found"
# ile HARNESS_ERROR döner ve hata **kırık paket** gibi görünür — oysa kırık olan
# ortamdır.
#
# İkisi de sabit + doğrulanmış. rtk'nin sürümü ve digest'i docker/go-sandbox/Dockerfile'ın
# sabitlediğiyle **aynı**: yani defter ile sandbox imajı kanıtlanabilir biçimde aynı
# binary'yi koşuyor. Katalogun çıktı formatı rtk'nin formatıdır; farklı bir build
# üzerinde ölçmek, eval harness'ın ölçeceği formatı ölçmemek demektir.
#
# rtk doğrulaması çift kaynaklı: gömülü sabit *ve* yayıncının checksums.txt'i. Uyuşmazsa
# kurulum iptal edilir.
#
# GOTOOLCHAIN=local: go.mod'daki `go 1.23` direktifinin ağdan toolchain indirmesini
# engeller — testlerin ürettiği modüller ağ olmadan da derlenir.

import hashlib
import io
import os
import pathlib
import platform
import shutil
import tarfile
import urllib.request

GO_VERSION = '1.23.6'
GO_SHA256 = '9379441ea310de000f33a4dc767bd966e72ab2826270e038e78b2c53c2e7802d'
GO_ROOT = '/usr/local/go'

RTK_VERSION = '0.50.0'
RTK_SHA256 = 'bc2b8902b0d9c796c82ef45f16ae2307e17757afeca5ee156235a3dc7bda5f89'
RTK_BASE_URL = 'https://github.com/rtk-ai/rtk/releases/download'
RTK_ASSET = 'rtk-x86_64-unknown-linux-musl.tar.gz'


def fetch_verified(url: str, expected_sha256: str, what: str) -> bytes:
    """İndir, hash'le, sabitle karşılaştır. Uyuşmazsa kurulum yapılmaz."""
    print(f"  indiriliyor {what}: {url}")
    with urllib.request.urlopen(url, timeout=300) as response:
        payload = response.read()
    digest = hashlib.sha256(payload).hexdigest()
    print(f"  sha256           {digest}")
    if digest != expected_sha256:
        raise SystemExit(
            f"{what} doğrulanamadı. İndirilen {digest}, beklenen {expected_sha256}. "
            "Kurulum iptal edildi — doğrulanmamış bir toolchain, hiç kurmamaktan kötüdür."
        )
    return payload


if platform.machine() not in ("x86_64", "AMD64"):
    raise SystemExit(
        f"bu defter {platform.machine()} mimarisi için; rtk'nin sabitlenmiş build'i "
        "x86_64. Colab'da T4/L4/A100 seçtiğiniz için bu burada sorun olmaz."
    )

# ------------------------------------------------------------------ Go
print("Go kuruluyor...")
go_archive = f"go{GO_VERSION}.linux-amd64.tar.gz"
go_payload = fetch_verified(f"https://go.dev/dl/{go_archive}", GO_SHA256, "go")

root = pathlib.Path(GO_ROOT)
if root.exists():
    shutil.rmtree(root)
with tarfile.open(fileobj=io.BytesIO(go_payload)) as tar:
    try:
        tar.extractall(root.parent, filter="data")
    except TypeError:  # Python < 3.12 has no filter argument
        tar.extractall(root.parent)

# ------------------------------------------------------------------ rtk
print("rtk kuruluyor...")
base = f"{RTK_BASE_URL}/v{RTK_VERSION}"
rtk_payload = fetch_verified(f"{base}/{RTK_ASSET}", RTK_SHA256, "rtk")

# İkinci kaynak: yayıncının kendi checksum listesi. Sabit "neyi inceledik"i söyler,
# bu indirme "yayıncı hâlâ öyle diyor"u kanıtlar.
with urllib.request.urlopen(f"{base}/checksums.txt", timeout=120) as response:
    published = response.read().decode("utf-8", "replace")
expected_line = next(
    (line for line in published.splitlines() if line.split()[-1:] == [RTK_ASSET]), None
)
if expected_line is None:
    raise SystemExit(f"checksums.txt içinde {RTK_ASSET} yok; yayıncı listeyi değiştirmiş.")
if expected_line.split()[0] != RTK_SHA256:
    raise SystemExit(
        f"sabit ile yayıncı uyuşmuyor. Gömülü {RTK_SHA256}, yayıncı "
        f"{expected_line.split()[0]}. İkisi de doğru olmalı."
    )
print(f"  yayıncı da doğruladı: {expected_line.split()[0][:16]}...")

rtk_dir = pathlib.Path("/tmp/rtk-dist")
if rtk_dir.exists():
    shutil.rmtree(rtk_dir)
rtk_dir.mkdir(parents=True)
with tarfile.open(fileobj=io.BytesIO(rtk_payload)) as tar:
    tar.extractall(rtk_dir)
binary = rtk_dir / "rtk"
shutil.copy2(binary, "/usr/local/bin/rtk")
os.chmod("/usr/local/bin/rtk", 0o755)
shutil.rmtree(rtk_dir, ignore_errors=True)

# ------------------------------------------------------------------ PATH
# os.environ üzerinden, shell export'u değil: subprocess os.environ'ı miras alır,
# dolayısıyla hem kalite kapısı hem eğitim koşusu aynı toolchain'i görür. Shell'e
# kurulup sürece aktarılmayan bir Go, kapıdan geçer ve eğitimde patlar.
os.environ["PATH"] = f"{root / 'bin'}:/usr/local/bin:{os.environ['PATH']}"
os.environ["GOTOOLCHAIN"] = "local"

# Parçalanma ayarı. CUDA önbelleğinde ayrılmış ama kullanılmayan bloklar uzun
# bir koşuda birikir ve "boş" görünen belleği yer. Bu ayar ayrılmış blokları
# büyüterek birleştirir; etkisi çalışma zamanında görülür, tahmin edilemez.
if EXPANDABLE_SEGMENTS:
    _existing = os.environ.get("PYTORCH_CUDA_ALLOC_CONF", "")
    if "expandable_segments" not in _existing:
        os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
            f"{_existing},expandable_segments:True" if _existing else "expandable_segments:True"
        )
    print("PYTORCH_CUDA_ALLOC_CONF:", os.environ["PYTORCH_CUDA_ALLOC_CONF"])

# `run` 3. hücreden.
print(run("go", "version"))
print(run("rtk", "--version"))

In [ ]:
# @title 4 — Tüm boru hattı, tek hucrede
#
# Buradaki tek hucre butun boru hattini calistiriyor: kalite kapisi, korpus, veri
# olcumu, SFT, degerlendirme, tercih ciftleri, DPO. Sira, kosullar ve hata
# durumlari `src/gotooltrain/pipeline.py` icinde; burada yalnizca ayarlar var.
#
# Neden tek hucre: bir defteri on bir hucreye bolmek, sirayi **okuyan kisinin
# hafizasina** yaziyor. Bir hucre dusunce sonrakiler yine calisiyor ve hangi
# asamanin gercekten gerceklestigi belirsizlesiyor. Burada her asamanin adi bastan
# yazilir, ciktisi canli akar ve **bir asama basarisiz olursa boru hatti durur** --
# yarisi calismis bir kosu, tamamlanmis gibi gorunmez.
#
# Her asamanin logu `runs/pipeline/<asama>.log` altinda. Cikti kaydirilinca teshis
# kaybolmaz; hata olursa o asamanin son satirlari hucrede basilir.
#
# Once `--dry-run` ile bir dene: komutlar yazilir, hicbir sey calismaz.
#
# Calisma suresi: kalite kapisi + korpus + SFT saatler; degerlendirme ve DPO bunlarin
# ustune eklenir. Yayinlar PUSH_EVERY adimda birer olur.

import subprocess
import sys

command = [
    sys.executable, "-u", "-m", "gotooltrain.pipeline",
    "--model", MODEL_ID,
    "--context-length", str(CONTEXT_LENGTH),
    "--memory-budget-gb", str(MEMORY_BUDGET_GB),
    "--optimizer", OPTIMIZER,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--learning-rate", str(LEARNING_RATE),
    "--max-records", str(MAX_RECORDS),
    "--sft-output", SFT_OUTPUT,
]

# Tercih zinciri, degerlendirme icin bir gorev dosyasi ve ornekleme yapacak calisan
# bir model ister. Yoksa bu asamalar **acikca** atlanir; sahte bir basari uretilmez.
if not RUN_PREFERENCE_STAGE:
    command += ["--no-eval", "--no-dpo"]

# Kalite kapisi her kosuda yeniden kurulmaz; tekrar etmek dakikalar verir ve
# sonucu degistirmez. Kapatmak bir tercih, sessizce yapilmaz.
if not RUN_QUALITY_GATE:
    command.append("--no-gate")

if RESUME_FROM:
    command += ["--extra-sft", "--resume-from", RESUME_FROM]

if DRY_RUN:
    command.append("--dry-run")
else:
    # Kuru kosuda hicbir sey yuklenmez, bu yuzden yayin bayraklari da gonderilmez:
    # dry-run komutu zaten hicbir sey gondermez, ama bayragi da hicbir seye yaramaz.
    command += ["--extra-sft", "--hub-repo-id", HF_REPO_ID, "--hub-push-every", str(PUSH_EVERY)]

print(" ".join(command))
print()

result = subprocess.run(command, check=False)
print()
print("Boru hatti bitti. Ozet ve loglar: runs/pipeline/")
if result.returncode:
    raise SystemExit(
        f"Boru hatti {result.returncode} ile durdu. Yukarida hangi asamanin ve neden "
        "durdugu yaziyor; o asamanin logu runs/pipeline/ altinda. Bu hucre hatayi "
        "yutmaz."
    )

## Bu koşu ne ölçer, ne ölçmez

**Ölçer:** token formatı doğrudan gerçek tokenizer ile üretildi; maske `{% generation %}`
bloğundan geldi; loss yalnızca asistan token'larına uygulandı; 4B tam FT gerçekten
bir optimizer adımı attı ve checkpoint yazdı; periyodik yayın takvimi çalıştı.

**Ölçmez:** Go yeteneği. Korpus Go-UT-Bench'in "bu dosya için birim testi yaz"
görevlerinden gelir - tek turlu, araçsız, sekiz depodan biri ağırlıklı. Bu, korpusun
katalog öretmediğini gösteren bir gerçektir, hata değil.

12-13. hücreler bir tercih aşaması sunar, ama **görev dosyası olmadan durur** ve bunu
söyler. Bunun sebebi yetenek eksikliği değil, girdi eksikliği: tercih çifti ancak
bir görevin `verification` komutu çalıştırılıp bir deneme geçerken bir başkası
kalırken doğar. Go-UT-Bench `(depo, dosya, kod)` verir; o katman
`src/gotooltrain/tasks.py`'nin işidir; mimari karar `docs/EVAL.md`'de. Sıfır çift
üretip "DPO çalıştı" demek, hiç karşılaştırma yapılmamışken başarı göstermek
olurdu — o yüzden hücre adıyla durur.

## Yayınlama hakkında dürüstlük

- `--hub-dry-run` **hiçbir şey yüklemez**; defter bunu log'da `DRY RUN` diye yazar.
- Gerçek koşuda `--hub-repo-id` verdiyseniz token'ın **yazma yetkisi** olmalı.
  Yazma yetkisi yoksa ilk push'ta hata verir — 700 adım sonra değil, `HF_TOKEN`
  çözümlendiği anda değil (pusher, model yüklenmeden önce kurulur).
- Yayınlanan klasör `hub_push.json` taşır: adım, toplam adım ve tüm hiperparametreler.
  Token **oraya yazılmaz**; sadece değişkenin adı yazılır.
- Bu bir **checkpoint**, tam bir **resume** değildir: optimizer durumu saklanmaz.
  Kaldığınız yerden devam etmek `resume_from` + yeniden eğitim demektir.
            